# 因果語言模型（Causal LM）訓練實例

## 學習目標

1. 理解因果語言模型（CLM）的自迴歸訓練目標，與遮罩語言模型（MLM）的差異。
2. 使用 `datasets` 3.x 管線：`load_dataset` + `map(batched=True)` + `DataCollatorForLanguageModeling`。
3. 以 2026 統一慣例載入模型：`device_map='auto'`、`torch_dtype=torch.bfloat16`、`use_safetensors=True`。
4. 以現代化 `TrainingArguments`（bf16、cosine scheduler、AdamW fused）搭配 `Trainer` 進行訓練。
5. 訓練後以 `pipeline(device_map='auto')` 執行文字生成推論，並示範 `push_to_hub`。

## 前置知識

- 基礎 Transformer 架構（attention、decoder-only）
- HuggingFace `AutoTokenizer` / `AutoModelForCausalLM` 基本用法
- 建議先完成 `02-Adv-tasks/05-retrieval_chatbot/retrieval_bot.ipynb`

## 銜接說明

- **前一個 notebook**：`../05-retrieval_chatbot/retrieval_bot.ipynb` — 檢索式對話機器人（向量相似度搜尋）
- **本 notebook**：從零預訓練一個中文 CLM，理解 next-token prediction 訓練目標
- **下一個 notebook**：`../07-text_summarization/summarization.ipynb` — 以 Seq2Seq 模型做摘要生成
- **延伸學習**：`../../03-PEFT/` — LoRA / QLoRA 指令微調；`../../04-kbits-tuning/` — BitsAndBytes 量化

## 環境準備：鎖定套件版本

以下版本組合經過 2026 驗證，確保 API 相容性。若在 Colab / Kaggle 等環境執行，請先執行此 cell。

In [ ]:
# Version lock — run once, then restart kernel
# %pip install -q \
#   "transformers>=4.46" \
#   "datasets>=3.0" \
#   "accelerate>=1.0" \
#   "safetensors>=0.4" \
#   "torch>=2.4" \
#   "evaluate>=0.4"

## Step 1：匯入套件

2026 慣例：
- `AutoModelForCausalLM` 取代具體類別（如 `BloomForCausalLM`），讓程式碼與模型無關。
- `DataCollatorForLanguageModeling(mlm=False)` 是 CLM 的標準 collator，會自動左移標籤（shift labels），不需手動處理。
- `set_seed` 確保訓練可重現。

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    pipeline,
    set_seed,
)

set_seed(42)

## Step 2：載入資料集

使用 HuggingFace Hub dataset id，可在任何環境直接重現，不依賴本機路徑。

> `wiki_cn_filtered` 是一份過濾後的中文維基百科語料，欄位為 `completion`（純文字）。
> 若 Hub 上無公開版本，可自行上傳後替換 `DATASET_ID`。
> 本機已有快取的情況下，`load_dataset` 不會重複下載。

In [ ]:
# Replace with your actual HuggingFace dataset id, or use load_from_disk() for local data.
# Example: DATASET_ID = "your-org/wiki_cn_filtered"
DATASET_ID = "wiki_cn_filtered"   # adjust as needed

try:
    ds = load_dataset(DATASET_ID, split="train")
except Exception:
    # Fallback: load from local disk if Hub id is not set up
    from datasets import Dataset
    ds = Dataset.load_from_disk("./wiki_cn_filtered/")

print(ds)
print(ds[0])

## Step 3：載入 Tokenizer 並定義前處理函式

**CLM 前處理重點**：
- 每筆樣本末尾加 `eos_token`，讓模型學習句子終止條件。
- `truncation=True` + `max_length=384`：BLOOM 的位置編碼支援長序列，但訓練時截斷可節省 VRAM。
- 不設定 `padding`：後續 `DataCollatorForLanguageModeling` 會做動態 padding，避免浪費算力在 `[PAD]` token 上。

**`batched=True` 的效益**：
- HuggingFace `map` 預設逐筆處理；`batched=True` 批次呼叫 tokenizer，速度快 3–5 倍。
- `remove_columns=ds.column_names` 移除原始文字欄位，只保留模型需要的 `input_ids` / `attention_mask`。

In [ ]:
MODEL_ID = "Langboat/bloom-389m-zh"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_fn(examples):
    # Append eos_token so the model learns when a document ends.
    texts = [text + tokenizer.eos_token for text in examples["completion"]]
    return tokenizer(
        texts,
        max_length=384,
        truncation=True,
        # No padding here; DataCollatorForLanguageModeling handles dynamic padding.
    )

tokenized_ds = ds.map(
    tokenize_fn,
    batched=True,
    num_proc=4,               # parallel CPU workers
    remove_columns=ds.column_names,
    desc="Tokenizing",
)
print(tokenized_ds)

## Step 4：理解 DataCollatorForLanguageModeling（CLM 模式）

`DataCollatorForLanguageModeling(tokenizer, mlm=False)` 在 CLM 模式下做三件事：

1. **動態 padding**：將 batch 內所有序列 pad 到最長長度（比固定 max_length 更省記憶體）。
2. **Label 左移（shift）**：`labels = input_ids.clone()`，模型內部在計算 cross-entropy 時自動 shift，不需手動處理。
3. **Pad token 遮罩**：`labels` 中對應 `pad_token_id` 的位置設為 `-100`，loss 計算時忽略。

> **CLM vs MLM**：
> - CLM（`mlm=False`）：給定前綴預測下一個 token（autoregressive，單向 attention）— 用於文字生成。
> - MLM（`mlm=True`）：隨機遮罩部分 token 並預測（雙向 attention）— 用於 BERT 系列。

以下用小型 DataLoader 驗證 collator 的輸出格式。

In [ ]:
from torch.utils.data import DataLoader

collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Inspect one batch to verify label shape and -100 masking on pad positions.
dl = DataLoader(tokenized_ds.select(range(8)), batch_size=2, collate_fn=collator)
batch = next(iter(dl))
print("input_ids shape :", batch["input_ids"].shape)
print("labels shape    :", batch["labels"].shape)
print("labels[0][:10] :", batch["labels"][0][:10])   # first 10 tokens
print("pad_token       :", tokenizer.pad_token, "id:", tokenizer.pad_token_id)
print("eos_token       :", tokenizer.eos_token, "id:", tokenizer.eos_token_id)

## Step 5：載入預訓練模型（2026 統一慣例）

```python
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
```

三個參數的說明：

| 參數 | 作用 |
|---|---|
| `device_map="auto"` | Accelerate 自動把模型分層分配到 GPU → CPU → Disk，不需手動 `.cuda()`；多 GPU 自動 tensor parallelism |
| `torch_dtype=torch.bfloat16` | bf16 的動態範圍與 fp32 相同（8 bit 指數），精度損失遠低於 fp16（只有 5 bit 指數，容易 overflow/underflow）；在 Ampere+ GPU（A100/RTX3090+）上與 fp32 幾乎等速，VRAM 節省 50% |
| `use_safetensors=True` | safetensors 格式相較 pickle 有安全性保證（不執行任意程式碼）、mmap 載入速度快 2–3 倍 |

> **VRAM 估算**（bloom-389m-zh）：bf16 約 780 MB，可在 8 GB 顯卡上舒適訓練。

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
)
print(f"Model dtype : {next(model.parameters()).dtype}")
print(f"Parameters  : {model.num_parameters() / 1e6:.1f} M")

## Step 6：設定 TrainingArguments（現代化版本）

以下是 2026 完整版本的說明：

| 參數 | 說明 |
|---|---|
| `bf16=True` | 以 bfloat16 做梯度計算，配合 `torch_dtype=bfloat16` 的模型，無需額外轉換 |
| `optim="adamw_torch_fused"` | PyTorch 2.x 原生 fused AdamW，比 `adam` 快 10–20%；AdamW 相較 Adam 有正確的 weight decay 實作 |
| `warmup_ratio=0.1` | 前 10% steps 線性 warmup，避免早期 loss spike，對小資料集尤其重要 |
| `lr_scheduler_type="cosine"` | cosine decay 使學習率平滑降低，優於 linear 在後期的急速下降 |
| `max_grad_norm=1.0` | 梯度裁剪，防止梯度爆炸 |
| `eval_strategy="steps"` | 周期性評估，搭配 `load_best_model_at_end=True` 可自動選最佳 checkpoint |
| `save_safetensors=True` | 儲存為安全的 safetensors 格式 |
| `seed=42` | 確保可重現性 |

> **Effective batch size** = `per_device_train_batch_size` × `gradient_accumulation_steps` × GPU 數量
> = 4 × 8 × 1 = **32**（在單 GPU 環境下等效 batch size 32，但只佔 1/8 的 VRAM）

In [ ]:
args = TrainingArguments(
    output_dir="./causal_lm",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,     # effective batch = 4 * 8 = 32
    num_train_epochs=1,
    logging_steps=10,
    # --- 2026 additions ---
    bf16=True,                          # use bfloat16; requires Ampere+ GPU (A100, RTX30xx+)
    optim="adamw_torch_fused",          # fused AdamW is faster and more memory-efficient
    warmup_ratio=0.1,                   # linear warmup for first 10% of steps
    lr_scheduler_type="cosine",         # smooth cosine decay
    max_grad_norm=1.0,                  # gradient clipping
    eval_strategy="no",                 # no eval split in this demo; set to 'steps' with eval_dataset
    save_strategy="epoch",
    save_safetensors=True,              # save in safetensors format
    seed=42,
    report_to="none",                   # disable wandb/tensorboard for demo
)

## Step 7：建立 Trainer 並訓練

`DataCollatorForLanguageModeling(tokenizer, mlm=False)` 已處理動態 padding 與標籤遮罩，Trainer 只需傳入 collator 即可，不需手刻訓練迴圈。

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)

trainer.train()

## Step 8：儲存模型與 Tokenizer

`save_pretrained(safe_serialization=True)` 確保輸出為 safetensors 格式。`push_to_hub` 示範如何將訓練成果上傳到 HuggingFace Hub，使任何人可直接使用。

> 若無 HuggingFace token，請先執行 `huggingface-cli login` 或設定 `HF_TOKEN` 環境變數。

In [ ]:
import os
from pathlib import Path

SAVE_DIR = Path("./causal_lm/final")

# Save locally in safetensors format.
trainer.model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR.resolve()}")

# (Optional) push to HuggingFace Hub.
# Requires: huggingface-cli login  OR  HF_TOKEN environment variable.
# HUB_REPO = "your-username/bloom-389m-zh-clm-finetuned"
# trainer.push_to_hub(HUB_REPO)

## Step 9：模型推論（文字生成）

```python
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device_map="auto")
```

`device_map="auto"` 與模型載入時的語意一致，Accelerate 自動決定最佳裝置配置，同時支援 CPU offload 與多 GPU。

**生成參數說明**：
- `do_sample=True`：隨機取樣，啟用 top-k / top-p 控制多樣性。
- `temperature=0.8`：softmax 溫度，數值越低輸出越確定（greedy），越高越多樣。
- `max_new_tokens`：只計算新生成的 token 數量，不把 prompt 計入。

In [ ]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
)

# --- Example 1: University museum ---
result1 = pipe(
    "西安交通大學博物館（Xi'an Jiaotong University Museum）是一座位於西安",
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
)
print(result1[0]["generated_text"])
print("-" * 60)

# --- Example 2: Gaming news ---
result2 = pipe(
    "下面是一則遊戲新聞。小編報道，近日，遊戲產業發展得非常",
    max_new_tokens=80,
    do_sample=True,
    temperature=0.8,
    top_p=0.92,
)
print(result2[0]["generated_text"])

## 小結

### 本 notebook 涵蓋的核心概念

1. **CLM 訓練目標**：next-token prediction，模型學習 P(x_t | x_1, ..., x_{t-1})，損失函式是整個序列上的 cross-entropy。
2. **`DataCollatorForLanguageModeling(mlm=False)`**：自動處理動態 padding 與標籤左移，不需手刻 `-100` 遮罩。
3. **2026 模型載入三件套**：`device_map='auto'` + `torch_dtype=bfloat16` + `use_safetensors=True`，一行搞定裝置、精度、安全格式。
4. **現代化 TrainingArguments**：bf16、fused AdamW、cosine scheduler、warmup、梯度裁剪，這些是訓練穩定性的基礎設定。
5. **可重現性與可移植性**：`set_seed(42)`、移除硬路徑、`save_safetensors=True`、`push_to_hub` 示範。

### 延伸練習

1. **分割訓練 / 驗證集**：將資料集以 `ds.train_test_split(test_size=0.05, seed=42)` 分割，在 `TrainingArguments` 設定 `eval_strategy="steps"`，觀察驗證 loss 曲線。
2. **調整 `max_new_tokens` 與 `temperature`**：比較 `temperature=0.3`（低多樣性）與 `temperature=1.2`（高多樣性）的生成品質差異。
3. **進階微調**：前往 `../../03-PEFT/` 學習如何用 LoRA 對本模型做指令微調，大幅降低 VRAM 需求。
4. **量化推論**：前往 `../../04-kbits-tuning/` 學習 4-bit / 8-bit 量化，在消費級 GPU（8 GB VRAM）上執行更大的模型。
5. **對話模板**：試試 `tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)`，體會為何統一對話模板比手寫 `[INST]` 更具可移植性，這是 `../../03-PEFT/` 指令微調的前置概念。